<a href="https://colab.research.google.com/github/Le2se0hy/FA_ProAn/blob/main/%EC%B5%9C%EC%A2%85%EB%B3%80%EC%88%98%EC%A1%B0%ED%95%A9%EC%B0%BE%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.neighbors import BallTree
from collections import OrderedDict
from itertools import combinations
import os

# ============================================================
# 0) 유틸
# ============================================================
def haversine_km_vec(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * (np.sin(dlon / 2.0) ** 2)
    c = 2.0 * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))
    return R * c

def rmse_fitlm(res):
    return float(np.sqrt(res.mse_resid))

def fit_model(df, y_col, x_cols):
    X = sm.add_constant(df[x_cols], has_constant="add")
    y = df[y_col]
    return sm.OLS(y, X, missing="drop").fit()

def get_sign(res, var, tol=1e-12):
    if var not in res.params.index:
        return 0
    v = float(res.params[var])
    if abs(v) <= tol:
        return 0
    return 1 if v > 0 else -1

# ============================================================
# 1) 중복 좌표 처리
# ============================================================
def make_unique_sum_count(lat, lon, y):
    df_loc = pd.DataFrame({"y": lat, "x": lon, "Y": y})
    g = df_loc.groupby(["y", "x"], sort=True)["Y"].agg(["sum", "count"]).reset_index()
    g["_uix"] = np.arange(len(g), dtype=int)

    lat_uni = g["y"].to_numpy(dtype=float)
    lon_uni = g["x"].to_numpy(dtype=float)
    sumY_uni = g["sum"].to_numpy(dtype=float)
    count_uni = g["count"].to_numpy(dtype=float)

    idx_map = df_loc[["y", "x"]].merge(
        g[["y", "x", "_uix"]],
        on=["y", "x"],
        how="left",
        sort=False
    )["_uix"].to_numpy(dtype=int)

    return lat_uni, lon_uni, sumY_uni, count_uni, idx_map

# ============================================================
# 2) WY 계산 (연도별 1회만)
# ============================================================
def compute_WY_unique_counts(lat_uni, lon_uni, sumY_uni, count_uni, distance_band_km=1.0, eps=1e-12):
    lat_r = np.deg2rad(lat_uni.astype(float))
    lon_r = np.deg2rad(lon_uni.astype(float))
    coords = np.column_stack([lat_r, lon_r])

    R = 6371.0
    rad_band = distance_band_km / R
    rad_band_candidate = rad_band * (1.0 + 1e-12)

    tree = BallTree(coords, metric="haversine")
    n = len(lat_uni)
    WY_uni = np.zeros(n, dtype=float)

    for i in range(n):
        idx = tree.query_radius(coords[i:i+1], r=rad_band_candidate, return_distance=False)[0]
        idx = idx[idx != i]

        if idx.size == 0:
            WY_uni[i] = 0.0
            continue

        d_km = haversine_km_vec(lat_r[i], lon_r[i], lat_r[idx], lon_r[idx])
        mask = (d_km <= distance_band_km + eps) & (d_km > 0)
        idx2 = idx[mask]
        d2 = d_km[mask]

        if idx2.size == 0:
            WY_uni[i] = 0.0
            continue

        invd = 1.0 / d2
        num = np.sum(invd * sumY_uni[idx2])
        den = np.sum(invd * count_uni[idx2])
        if den == 0:
            den = 1.0

        WY_uni[i] = num / den

    return WY_uni

# ============================================================
# 3) rho 탐색
# ============================================================
def search_best_rho(df, y_col, x_cols, WY, rho_grid=None):
    if rho_grid is None:
        rho_grid = np.round(np.arange(-0.90, 0.90 + 1e-12, 0.05), 2)

    df_tmp = df.copy()
    df_tmp["_WY_"] = WY

    best_rho = float(rho_grid[0])
    df_tmp["_Y_SPLAG_"] = df_tmp[y_col] - best_rho * df_tmp["_WY_"]
    res_best = fit_model(df_tmp, "_Y_SPLAG_", x_cols)
    best_rmse = rmse_fitlm(res_best)

    for rho in rho_grid[1:]:
        rho = float(rho)
        df_tmp["_Y_SPLAG_"] = df_tmp[y_col] - rho * df_tmp["_WY_"]
        res = fit_model(df_tmp, "_Y_SPLAG_", x_cols)
        r = rmse_fitlm(res)
        if r < best_rmse:
            best_rho, best_rmse, res_best = rho, float(r), res

    return best_rho, best_rmse, res_best

# ============================================================
# 4) 표 작성용
# ============================================================
def mark_sig(res, var, digits=3):
    if var not in res.params.index:
        return "–"
    coef = res.params[var]
    pval = res.pvalues[var]
    s = f"{coef:.{digits}f}"

    if pval < 0.01:
        s += "‡"
    elif pval < 0.05:
        s += "†"
    return s

def build_table(results_dict, col_order, row_map, obs, f_round_to_10=False):
    T = pd.DataFrame(index=list(row_map.keys()), columns=col_order)

    for col in col_order:
        res = results_dict[col]

        for rname, vname in row_map.items():
            if vname is None:
                T.loc[rname, col] = ""
                continue
            if vname in ["F", "RMSE", "AdjR2"]:
                continue
            T.loc[rname, col] = mark_sig(res, vname)

        fval, fp = res.fvalue, res.f_pvalue
        if fval is None:
            ftxt = "–"
        else:
            fnum = round(fval, -1) if f_round_to_10 else round(fval)
            ftxt = f"{fnum:,.0f}"
            if fp < 0.01:
                ftxt += "‡"
            elif fp < 0.05:
                ftxt += "†"
            elif fp < 0.10:
                ftxt += "+"

        T.loc["F-statistics", col] = ftxt
        T.loc["RMSE", col] = f"{rmse_fitlm(res):.3f}"
        T.loc["Adjusted $R^2$", col] = f"{res.rsquared_adj:.3f}"

    header = f"Obs.= {obs:,}"
    return header, T

# ============================================================
# 5) 파일 전처리
# ============================================================
def prepare_df(excel_path):
    df = pd.read_excel(excel_path)

    rename_map = {
        # y
        "Log_Price_per_m2": "Price",

        # 위치
        "Latitude": "y",
        "Longitude": "x",

        # 기본 구조
        "Size_m2": "Area",
        "Construction_Year": "Year",
        "Floor": "Floor",
        "max_floor": "MaxFloor",
        "Parking_per_Household": "Parking",

        # 거리/입지
        "Log_Dist_Water": "Dist. Water",
        "Log_Dist_Green": "Dist. Green",
        "Log_Dist_Subway": "Dist. Subway",
        "Dist_CBD": "Dist. CBD",

        # 지역/사회경제
        "Population": "Population",
        "Sex_ratio": "Sex_ratio",
        "Pop. Density": "Pop. Density",
        "Old Population": "Old Population",
        "Median age": "Median age",
        "Young Population": "Young Population",
        "num_of_people": "Households",

        # 생활 인프라
        "Bus_Stop": "Bus Stop",
        "High_School_Count": "High School Cnt",

        # 연도/월
        "Year_Sold": "SaleYear",
        "Month_Sold": "Month",
    }

    for k, v in rename_map.items():
        if k in df.columns and v not in df.columns:
            df = df.rename(columns={k: v})

    # heating: 도시가스=1, 아니면 0
    if "heating" in df.columns and "Heating" not in df.columns:
        # 이미 0/1이면 그대로 처리 가능하게 함
        if pd.api.types.is_numeric_dtype(df["heating"]):
            vals = pd.to_numeric(df["heating"], errors="coerce").fillna(0)
            # 0/1이라고 가정
            df["Heating"] = (vals != 0).astype(int)
        else:
            df["Heating"] = (df["heating"].astype(str).str.contains("도시가스", na=False)).astype(int)

    # 계절 더미 생성 (항상 모델에 포함될 예정)
    if "Month" in df.columns:
        m = pd.to_numeric(df["Month"], errors="coerce")
        df["Spring"] = m.isin([3, 4, 5]).astype(int)
        df["Fall"]   = m.isin([9, 10, 11]).astype(int)
        df["Winter"] = m.isin([12, 1, 2]).astype(int)
    else:
        df["Spring"] = 0
        df["Fall"] = 0
        df["Winter"] = 0

    return df

# ============================================================
# 6) 연도별 기본 데이터 생성 (WY 1회 계산)
# ============================================================
def prepare_year_base(df_year, distance_band=1.0):
    df = df_year.copy()

    numeric_candidates = [
        "Price", "y", "x",
        "Area", "Floor", "MaxFloor", "Parking", "Heating", "Year",
        "Dist. Water", "Dist. Green", "Dist. Subway", "Dist. CBD",
        "Population", "Sex_ratio", "Pop. Density",
        "Old Population", "Median age", "Young Population",
        "Households", "Bus Stop", "High School Cnt",
        "Spring", "Fall", "Winter"
    ]

    for c in numeric_candidates:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df_wy = df[["Price", "y", "x"]].dropna().copy()

    Y = df_wy["Price"].to_numpy(dtype=float)
    lat = df_wy["y"].to_numpy(dtype=float)
    lon = df_wy["x"].to_numpy(dtype=float)

    lat_uni, lon_uni, sumY_uni, count_uni, idx_map = make_unique_sum_count(lat, lon, Y)
    WY_uni = compute_WY_unique_counts(
        lat_uni, lon_uni, sumY_uni, count_uni,
        distance_band_km=distance_band
    )
    df_wy["IND_spw"] = WY_uni[idx_map]

    # 주의: Price, y, x 중복 조합이 있을 수 있으므로 index 기준으로 붙이지 않고 merge 사용
    df = df.merge(
        df_wy[["Price", "y", "x", "IND_spw"]],
        on=["Price", "y", "x"],
        how="left"
    )

    return df

# ============================================================
# 7) 한 조합 평가
#    - seasonal_vars는 항상 포함
#    - sign_check_vars만 부호 비교
# ============================================================
def evaluate_one_subset(df_base, x_cols, sign_check_vars, rho_grid=None):
    df = df_base.copy()

    needed_cols = ["Price", "IND_spw"] + x_cols
    needed_cols = [c for c in needed_cols if c in df.columns]
    df = df[needed_cols].dropna().copy()

    if len(df) < 50:
        return None

    try:
        res_ols = fit_model(df, "Price", x_cols)
    except:
        return None

    try:
        rho_best, rmse_net, _ = search_best_rho(
            df, "Price", x_cols, df["IND_spw"].to_numpy(), rho_grid=rho_grid
        )
        df["Y_splag_net"] = df["Price"] - rho_best * df["IND_spw"]
        res_slr = fit_model(df, "Y_splag_net", x_cols)
    except:
        return None

    sign_match_count = 0
    sign_all_match = True
    sign_detail = {}

    for v in sign_check_vars:
        s_ols = get_sign(res_ols, v)
        s_slr = get_sign(res_slr, v)
        same = (s_ols == s_slr) and (s_ols != 0)

        if same:
            sign_match_count += 1
        else:
            sign_all_match = False

        sign_detail[v] = {
            "OLS": s_ols,
            "SLR": s_slr,
            "same": same
        }

    return {
        "x_cols": x_cols,
        "n": len(df),
        "rho_best": rho_best,
        "rmse_net": rmse_net,
        "ols_rmse": rmse_fitlm(res_ols),
        "slr_rmse": rmse_fitlm(res_slr),
        "ols_adj_r2": res_ols.rsquared_adj,
        "slr_adj_r2": res_slr.rsquared_adj,
        "sign_match_count": sign_match_count,
        "sign_all_match": sign_all_match,
        "sign_detail": sign_detail,
        "res_ols": res_ols,
        "res_slr": res_slr,
        "df_used": df.copy()
    }

# ============================================================
# 8) 연도별 조합 탐색
# ============================================================
def search_subsets_for_year(
    df_base,
    base_required_vars,
    seasonal_vars,
    sign_check_vars,
    optional_vars,
    top_n=5,
    max_k=3,
    forbid_pair=True,
    rho_grid=None
):
    results = []

    # 모델에는 항상 들어가는 변수
    always_included = base_required_vars + seasonal_vars

    for k in range(max_k + 1):
        for subset in combinations(optional_vars, k):
            subset = list(subset)

            # Floor / MaxFloor 동시포함 금지 옵션
            if forbid_pair and ("Floor" in subset) and ("MaxFloor" in subset):
                continue

            x_cols = always_included + subset

            out = evaluate_one_subset(
                df_base=df_base,
                x_cols=x_cols,
                sign_check_vars=sign_check_vars,
                rho_grid=rho_grid
            )

            if out is not None:
                results.append(out)

    results = sorted(
        results,
        key=lambda z: (
            -z["sign_match_count"],
            z["slr_rmse"],
            z["ols_rmse"],
            -z["slr_adj_r2"],
            len(z["x_cols"])
        )
    )

    return results[:top_n], results

# ============================================================
# 9) 최적 조합 표 생성
# ============================================================
def build_tables_from_best_result(best_result, f_round_to_10=True):
    df = best_result["df_used"].copy()
    x_cols = best_result["x_cols"]

    res_ols = {"(Full)": best_result["res_ols"]}
    res_slr = {"(Full)": best_result["res_slr"]}

    obs = int(df[["Price"] + x_cols].dropna().shape[0])

    row_map = OrderedDict([
        ("Property characteristics", None),
        ("Size", "Area"),
        ("Floor", "Floor"),
        ("Parking", "Parking"),
        ("Heating(dummy)", "Heating"),
        ("Year built", "Year"),
        ("Max floor", "MaxFloor"),

        ("Accessibility / environment", None),
        ("Dist. subway", "Dist. Subway"),
        ("Dist. CBD", "Dist. CBD"),
        ("Dist. green", "Dist. Green"),
        ("Dist. water", "Dist. Water"),

        ("Demographic context", None),
        ("Population", "Population"),
        ("Sex ratio", "Sex_ratio"),
        ("Pop. density", "Pop. Density"),
        ("Old population", "Old Population"),
        ("Median age", "Median age"),
        ("Young population", "Young Population"),
        ("Households", "Households"),

        ("Local context", None),
        ("Bus stops", "Bus Stop"),
        ("High school cnt", "High School Cnt"),

        ("Seasonality control", None),
        ("Spring", "Spring"),
        ("Fall", "Fall"),
        ("Winter", "Winter"),

        ("F-statistics", "F"),
        ("RMSE", "RMSE"),
        ("Adjusted $R^2$", "AdjR2"),
    ])

    h_ols, t_ols = build_table(res_ols, ["(Full)"], row_map, obs, f_round_to_10=f_round_to_10)
    h_slr, t_slr = build_table(res_slr, ["(Full)"], row_map, obs, f_round_to_10=f_round_to_10)

    return {
        "header_ols": h_ols,
        "table_ols": t_ols,
        "header_slr": h_slr,
        "table_slr": t_slr
    }

# ============================================================
# 10) 메인 실행 함수
# ============================================================
def run_by_year_with_sign_matching(
    excel_path,
    years=(2022,),
    distance_band=1.0,
    top_n=5,
    max_k=3,
    f_round_to_10=True,
    print_summary=True,
    rho_grid=None
):
    df_all = prepare_df(excel_path)

    if "SaleYear" not in df_all.columns:
        raise ValueError(
            "거래연도 컬럼 Year_Sold가 엑셀에 있어야 하고, prepare_df에서 SaleYear로 바뀝니다.\n"
            f"현재 컬럼: {list(df_all.columns)}"
        )

    # 부호를 맞춰야 하는 필수 변수
    sign_check_vars = [
        "Area", "Parking", "Year",
        "Dist. Subway", "Dist. Green", "Dist. Water"
    ]

    # 모델에는 반드시 포함되지만 부호 일치 검사 대상은 아님
    seasonal_vars = ["Spring", "Fall", "Winter"]

    # 모델에 항상 들어가야 하는 비계절 필수 변수
    base_required_vars = [
        "Area", "Parking", "Year",
        "Dist. Subway", "Dist. Green", "Dist. Water"
    ]

    # 선택 변수
    optional_vars = [
        "Floor",
        "MaxFloor",
        "Heating",
        "Dist. CBD",
        "Population",
        "Sex_ratio",
        "Pop. Density",
        "Old Population",
        "Median age",
        "Young Population",
        "Households",
        "Bus Stop",
        "High School Cnt",
    ]

    if rho_grid is None:
        rho_grid = np.round(np.arange(-0.90, 0.90 + 1e-12, 0.05), 2)

    out = {}

    for y in years:
        df_y = df_all[pd.to_numeric(df_all["SaleYear"], errors="coerce") == y].copy()

        print("\n" + "=" * 100)
        print(f"[{y}년] 데이터 개수 N = {len(df_y):,}")
        print("=" * 100)

        print("연도별 WY 계산 중...")
        df_base = prepare_year_base(df_y, distance_band=distance_band)

        print("변수 조합 탐색 중...")
        top_results, all_results = search_subsets_for_year(
            df_base=df_base,
            base_required_vars=base_required_vars,
            seasonal_vars=seasonal_vars,
            sign_check_vars=sign_check_vars,
            optional_vars=optional_vars,
            top_n=top_n,
            max_k=max_k,
            forbid_pair=True,
            rho_grid=rho_grid
        )

        if len(all_results) == 0:
            print("유효한 조합을 찾지 못했습니다.")
            out[y] = None
            continue

        best_result = all_results[0]

        if print_summary:
            print("\n[상위 조합 탐색 결과]")
            for i, r in enumerate(top_results, start=1):
                print(f"\n{i}. x_cols = {r['x_cols']}")
                print(f"   sign_match_count = {r['sign_match_count']} / {len(sign_check_vars)}")
                print(f"   sign_all_match   = {r['sign_all_match']}")
                print(f"   rho_best         = {r['rho_best']:.2f}")
                print(f"   OLS RMSE         = {r['ols_rmse']:.6f}")
                print(f"   SLR RMSE         = {r['slr_rmse']:.6f}")
                print(f"   SLR AdjR2        = {r['slr_adj_r2']:.6f}")
                print(f"   sign_detail      = {r['sign_detail']}")

        tables = build_tables_from_best_result(best_result, f_round_to_10=f_round_to_10)

        print("\n[최종 선택된 변수 조합]")
        print(best_result["x_cols"])
        print(f"필수변수 부호 일치 개수 = {best_result['sign_match_count']} / {len(sign_check_vars)}")
        print(f"Best rho = {best_result['rho_best']}")
        print(f"OLS RMSE = {best_result['ols_rmse']}")
        print(f"SLR RMSE = {best_result['slr_rmse']}")

        print("\nTable (OLS):", tables["header_ols"])
        print(tables["table_ols"])

        print("\nTable (SLR):", tables["header_slr"])
        print(tables["table_slr"])

        out[y] = {
            "best_result": best_result,
            "top_results": top_results,
            "all_results": all_results,
            "tables": tables,
            "df_base": df_base
        }

    return out

# ============================================================
# 11) 엑셀 저장 함수
# ============================================================
def save_sign_matching_results_to_excel(
    outs,
    out_path="Sign_Matching_Results.xlsx"
):
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    summary_rows = []

    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for year, pack in outs.items():
            if pack is None:
                continue

            best = pack["best_result"]
            top_results = pack["top_results"]
            tables = pack["tables"]

            summary_rows.append({
                "Year": year,
                "N_used": best["n"],
                "Best_rho": best["rho_best"],
                "Sign_Match_Count": best["sign_match_count"],
                "Sign_All_Match": best["sign_all_match"],
                "OLS_RMSE": best["ols_rmse"],
                "SLR_RMSE": best["slr_rmse"],
                "OLS_Adj_R2": best["ols_adj_r2"],
                "SLR_Adj_R2": best["slr_adj_r2"],
                "Selected_Variables": ", ".join(best["x_cols"])
            })

            top_rows = []
            for rank, r in enumerate(top_results, start=1):
                row = {
                    "Rank": rank,
                    "Variables": ", ".join(r["x_cols"]),
                    "N_used": r["n"],
                    "rho_best": r["rho_best"],
                    "sign_match_count": r["sign_match_count"],
                    "sign_all_match": r["sign_all_match"],
                    "OLS_RMSE": r["ols_rmse"],
                    "SLR_RMSE": r["slr_rmse"],
                    "OLS_Adj_R2": r["ols_adj_r2"],
                    "SLR_Adj_R2": r["slr_adj_r2"],
                }

                for v, d in r["sign_detail"].items():
                    row[f"{v}_OLS"] = d["OLS"]
                    row[f"{v}_SLR"] = d["SLR"]
                    row[f"{v}_same"] = d["same"]

                top_rows.append(row)

            pd.DataFrame(top_rows).to_excel(writer, sheet_name=f"Top_{year}", index=False)
            tables["table_ols"].to_excel(writer, sheet_name=f"OLS_{year}")
            tables["table_slr"].to_excel(writer, sheet_name=f"SLR_{year}")

            sign_rows = []
            for v, d in best["sign_detail"].items():
                sign_rows.append({
                    "Variable": v,
                    "OLS_sign": d["OLS"],
                    "SLR_sign": d["SLR"],
                    "Same?": d["same"]
                })
            pd.DataFrame(sign_rows).to_excel(writer, sheet_name=f"Sign_{year}", index=False)

            coef_rows = []
            res_ols = best["res_ols"]
            res_slr = best["res_slr"]
            all_vars = sorted(set(res_ols.params.index).union(set(res_slr.params.index)))

            for v in all_vars:
                coef_rows.append({
                    "Variable": v,
                    "OLS_coef": res_ols.params[v] if v in res_ols.params.index else np.nan,
                    "OLS_pvalue": res_ols.pvalues[v] if v in res_ols.pvalues.index else np.nan,
                    "SLR_coef": res_slr.params[v] if v in res_slr.params.index else np.nan,
                    "SLR_pvalue": res_slr.pvalues[v] if v in res_slr.pvalues.index else np.nan,
                })

            pd.DataFrame(coef_rows).to_excel(writer, sheet_name=f"Coef_{year}", index=False)

        pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

    print(f"엑셀 저장 완료: {out_path}")

# ============================================================
# 12) 실행 예시
# ============================================================
if __name__ == "__main__":
    outs = run_by_year_with_sign_matching(
        excel_path="!Seoul_Aprtment_FINAL.xlsx",
        years=(2022,),
        distance_band=1.0,
        top_n=5,
        max_k=3,
        f_round_to_10=True,
        print_summary=True,
        rho_grid=np.round(np.arange(-0.90, 0.90 + 1e-12, 0.05), 2)
    )

    save_sign_matching_results_to_excel(
        outs,
        out_path="Sign_Matching_2022.xlsx"
    )


[2022년] 데이터 개수 N = 10,579
연도별 WY 계산 중...
변수 조합 탐색 중...

[상위 조합 탐색 결과]

1. x_cols = ['Area', 'Parking', 'Year', 'Dist. Subway', 'Dist. Green', 'Dist. Water', 'Spring', 'Fall', 'Winter', 'Heating', 'Population', 'Bus Stop']
   sign_match_count = 6 / 6
   sign_all_match   = True
   rho_best         = 0.70
   OLS RMSE         = 0.389697
   SLR RMSE         = 0.318174
   SLR AdjR2        = 0.134292
   sign_detail      = {'Area': {'OLS': 1, 'SLR': 1, 'same': True}, 'Parking': {'OLS': 1, 'SLR': 1, 'same': True}, 'Year': {'OLS': 1, 'SLR': 1, 'same': True}, 'Dist. Subway': {'OLS': -1, 'SLR': -1, 'same': True}, 'Dist. Green': {'OLS': 1, 'SLR': 1, 'same': True}, 'Dist. Water': {'OLS': -1, 'SLR': -1, 'same': True}}

2. x_cols = ['Area', 'Parking', 'Year', 'Dist. Subway', 'Dist. Green', 'Dist. Water', 'Spring', 'Fall', 'Winter', 'Heating', 'Population', 'Pop. Density']
   sign_match_count = 6 / 6
   sign_all_match   = True
   rho_best         = 0.70
   OLS RMSE         = 0.390375
   SLR RMSE      

In [5]:
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

# ============================================================
# A. 엑셀 출력용 유의성 표시 함수
# ============================================================
def coef_with_sig(res, var, digits=4):
    if var not in res.params.index:
        return ""
    coef = res.params[var]
    pval = res.pvalues[var]

    s = f"{coef:.{digits}f}"
    if pval < 0.01:
        s += "‡"   # double dagger
    elif pval < 0.05:
        s += "†"   # dagger
    return s

# ============================================================
# B. 한 조합의 OLS/SLR 결과표 만들기
# ============================================================
def build_combo_result_table(result, digits=4):
    res_ols = result["res_ols"]
    res_slr = result["res_slr"]
    x_cols = result["x_cols"]

    preferred_order = [
        "const",
        "Area", "Floor", "Parking", "Heating", "Year", "MaxFloor",
        "Dist. Subway", "Dist. CBD", "Dist. Green", "Dist. Water",
        "Population", "Sex_ratio", "Pop. Density",
        "Old Population", "Median age", "Young Population",
        "Households", "Bus Stop", "High School Cnt",
        "Spring", "Fall", "Winter"
    ]

    vars_in_model = ["const"] + x_cols
    ordered_vars = [v for v in preferred_order if v in vars_in_model]
    ordered_vars += [v for v in vars_in_model if v not in ordered_vars]

    rows = []
    for v in ordered_vars:
        rows.append({
            "Variable": v,
            "OLS": coef_with_sig(res_ols, v, digits=digits),
            "SLR": coef_with_sig(res_slr, v, digits=digits),
            "OLS_coef_raw": res_ols.params[v] if v in res_ols.params.index else np.nan,
            "OLS_pvalue": res_ols.pvalues[v] if v in res_ols.pvalues.index else np.nan,
            "SLR_coef_raw": res_slr.params[v] if v in res_slr.params.index else np.nan,
            "SLR_pvalue": res_slr.pvalues[v] if v in res_slr.pvalues.index else np.nan,
        })

    rows.append({
        "Variable": "Adjusted R^2",
        "OLS": f"{res_ols.rsquared_adj:.4f}",
        "SLR": f"{res_slr.rsquared_adj:.4f}",
        "OLS_coef_raw": np.nan,
        "OLS_pvalue": np.nan,
        "SLR_coef_raw": np.nan,
        "SLR_pvalue": np.nan,
    })
    rows.append({
        "Variable": "RMSE",
        "OLS": f"{rmse_fitlm(res_ols):.4f}",
        "SLR": f"{rmse_fitlm(res_slr):.4f}",
        "OLS_coef_raw": np.nan,
        "OLS_pvalue": np.nan,
        "SLR_coef_raw": np.nan,
        "SLR_pvalue": np.nan,
    })
    rows.append({
        "Variable": "Best rho",
        "OLS": "",
        "SLR": f"{result['rho_best']:.4f}",
        "OLS_coef_raw": np.nan,
        "OLS_pvalue": np.nan,
        "SLR_coef_raw": np.nan,
        "SLR_pvalue": np.nan,
    })
    rows.append({
        "Variable": "N used",
        "OLS": f"{result['n']:,}",
        "SLR": f"{result['n']:,}",
        "OLS_coef_raw": np.nan,
        "OLS_pvalue": np.nan,
        "SLR_coef_raw": np.nan,
        "SLR_pvalue": np.nan,
    })
    rows.append({
        "Variable": "Sign match count",
        "OLS": f"{result['sign_match_count']}",
        "SLR": f"{result['sign_match_count']}",
        "OLS_coef_raw": np.nan,
        "OLS_pvalue": np.nan,
        "SLR_coef_raw": np.nan,
        "SLR_pvalue": np.nan,
    })

    return pd.DataFrame(rows)

# ============================================================
# C. 엑셀 서식 함수
# ============================================================
def style_worksheet(ws):
    header_fill = PatternFill(fill_type="solid", fgColor="D9EAF7")
    title_fill = PatternFill(fill_type="solid", fgColor="B4C7E7")
    thin = Side(style="thin", color="999999")

    for row in ws.iter_rows():
        for cell in row:
            cell.alignment = Alignment(vertical="center")

    for row in ws.iter_rows():
        for cell in row:
            if cell.value is not None:
                cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # 열 너비 자동 조정
    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = 0
        for cell in col_cells:
            try:
                val = str(cell.value) if cell.value is not None else ""
                max_len = max(max_len, len(val))
            except:
                pass
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 2, 40)

    # 제목/헤더 스타일
    for row in ws.iter_rows():
        for cell in row:
            if isinstance(cell.value, str):
                if cell.value.startswith("Top ") and "Combination" in cell.value:
                    cell.font = Font(bold=True, size=12)
                    cell.fill = title_fill
                elif cell.row > 1 and cell.value in [
                    "Item", "Value", "Variable", "OLS", "SLR",
                    "OLS_coef_raw", "OLS_pvalue", "SLR_coef_raw", "SLR_pvalue"
                ]:
                    cell.font = Font(bold=True)
                    cell.fill = header_fill

# ============================================================
# D. 통합 엑셀 저장 함수
#    - Summary 시트
#    - TopTables_2022 시트: 상위 조합별 표 블록
#    - Signs_2022 시트: 부호 비교
# ============================================================
def save_matching_excel_pretty(
    outs,
    out_path="2022_matching.xlsx",
    digits=4
):
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        summary_rows = []

        for year, pack in outs.items():
            if pack is None:
                continue

            top_results = pack["top_results"]
            best_result = pack["best_result"]

            # --------------------------
            # Summary
            # --------------------------
            summary_rows.append({
                "Year": year,
                "Best Variables": ", ".join(best_result["x_cols"]),
                "Sign Match Count": best_result["sign_match_count"],
                "All Match": best_result["sign_all_match"],
                "OLS Adjusted R^2": round(best_result["ols_adj_r2"], 4),
                "SLR Adjusted R^2": round(best_result["slr_adj_r2"], 4),
                "OLS RMSE": round(best_result["ols_rmse"], 4),
                "SLR RMSE": round(best_result["slr_rmse"], 4),
                "Best rho": round(best_result["rho_best"], 4),
                "N used": best_result["n"]
            })

            # --------------------------
            # 상위 조합별 표 시트
            # --------------------------
            sheet_name = f"TopTables_{year}"
            start_row = 0

            for rank, result in enumerate(top_results, start=1):
                info_df = pd.DataFrame([
                    ["Rank", rank],
                    ["Variables", ", ".join(result["x_cols"])],
                    ["Sign match count", result["sign_match_count"]],
                    ["All match", result["sign_all_match"]],
                    ["OLS Adjusted R^2", round(result["ols_adj_r2"], 4)],
                    ["SLR Adjusted R^2", round(result["slr_adj_r2"], 4)],
                    ["OLS RMSE", round(result["ols_rmse"], 4)],
                    ["SLR RMSE", round(result["slr_rmse"], 4)],
                    ["Best rho", round(result["rho_best"], 4)],
                    ["N used", result["n"]],
                ], columns=["Item", "Value"])

                table_df = build_combo_result_table(result, digits=digits)
                title_df = pd.DataFrame([[f"Top {rank} Combination"]], columns=["Top Combination"])

                title_df.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    startrow=start_row,
                    startcol=0,
                    index=False
                )

                info_df.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    startrow=start_row + 2,
                    startcol=0,
                    index=False
                )

                table_df.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    startrow=start_row + 2,
                    startcol=4,
                    index=False
                )

                start_row += max(len(info_df), len(table_df)) + 7

            # --------------------------
            # 부호 비교 시트
            # --------------------------
            sign_rows = []
            for rank, result in enumerate(top_results, start=1):
                for v, d in result["sign_detail"].items():
                    sign_rows.append({
                        "Rank": rank,
                        "Variable": v,
                        "OLS_sign": d["OLS"],
                        "SLR_sign": d["SLR"],
                        "Same": d["same"],
                        "Variables": ", ".join(result["x_cols"])
                    })

            pd.DataFrame(sign_rows).to_excel(
                writer,
                sheet_name=f"Signs_{year}",
                index=False
            )

        # Summary 시트 저장
        pd.DataFrame(summary_rows).to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        # 저장 후 서식 적용
        workbook = writer.book
        for ws_name in workbook.sheetnames:
            ws = workbook[ws_name]
            style_worksheet(ws)



    print(f"엑셀 저장 완료: {out_path}")

In [6]:
save_matching_excel_pretty(
    outs,
    out_path="2022_matching.xlsx",
    digits=4
)

엑셀 저장 완료: 2022_matching.xlsx
